In [1]:
from pyspark.sql import SparkSession
import os

# Nạp đầy đủ Package cho Kafka, Iceberg, Nessie và S3 (MinIO)
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1,'
    'org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.3.1,'
    'org.projectnessie.nessie-integrations:nessie-spark-extensions-3.4_2.12:0.67.0,'
    'org.apache.hadoop:hadoop-aws:3.3.4 '  # THƯ VIỆN CÒN THIẾU Ở ĐÂY
    'pyspark-shell'
)

spark = SparkSession.builder \
    .appName("Fix-S3A-Error") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://silver/") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# 2. Đọc bảng bằng Spark SQL
df = spark.sql("SELECT * FROM nessie.bronze_db.amazon_purchase_raw")

# 3. Chuyển đổi sang Pandas và hiển thị (đẹp hơn show())
# Lấy 10 dòng đầu tiên để xem nhanh
pdf = df.limit(5).toPandas()


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found

In [2]:
pdf

,kafka_value,kafka_timestamp
0,"{""Order Date"": ""2018-12-04"", ""Purchase Price P...",2026-04-26 08:53:46.548
1,"{""Order Date"": ""2018-12-22"", ""Purchase Price P...",2026-04-26 08:53:46.549
2,"{""Order Date"": ""2018-12-24"", ""Purchase Price P...",2026-04-26 08:53:46.550
3,"{""Order Date"": ""2018-12-25"", ""Purchase Price P...",2026-04-26 08:53:46.551
4,"{""Order Date"": ""2018-12-25"", ""Purchase Price P...",2026-04-26 08:53:46.560


In [3]:
from pyspark.sql.functions import count

# Tổng số dòng
total_count = df.count()

# Số dòng distinct
distinct_count = df.distinct().count()

print("Total rows:", total_count)
print("Distinct rows:", distinct_count)
print("Duplicate rows:", total_count - distinct_count)

Total rows: 1850717
Distinct rows: 1844334
Duplicate rows: 6383


In [4]:
from pyspark.sql.functions import col

duplicates = (
    df.groupBy(df.columns)
      .count()
      .filter(col("count") > 1)
      .orderBy(col("count").desc())
)

# Chỉ lấy 50 dòng đầu rồi mới chuyển
pdf = duplicates.limit(10).toPandas()

pdf

,order_date,unit_price,quantity,state,product_title,product_code,category,survey_id,processed_at,count
